In [18]:
# ============================================================
# H9: HIERARCHICAL LABEL-AWARE PROTOTYPE +
#     SUPERVISED CONTRASTIVE +
#     ORDINAL HYPERNETWORK
#
# ============================================================
#
# H9 PURPOSE
# ------------------------------------------------------------
#
# H8 demonstrated:
#
#   - higher overall accuracy
#   - somewhat improved macro-F1
#   - better Engage performance
#   - BUT Highly Disengage remains extremely weak
#
# H9 introduces a hierarchical classifier:
#
#
#                     ENGAGEMENT
#                         |
#              +----------+----------+
#              |                     |
#         DISENGAGED               ENGAGED
#              |                     |
#          +---+---+             +---+---+
#          |       |             |       |
#         HD       D             E       HE
#
#
# Instead of forcing one classifier to directly solve:
#
#       HD vs D vs E vs HE
#
# the model learns:
#
#   Stage 1:
#       disengaged vs engaged
#
#   Stage 2:
#       if disengaged:
#           HD vs D
#
#       if engaged:
#           E vs HE
#
#
# H9 RETAINS:
#
#   - label-aware support encoding
#   - HyperNetwork
#   - class prototypes
#   - supervised contrastive learning
#   - ordinal regularization
#   - class-aware episodic training
#
#
# H9 TEST PROTOCOL:
#
#   UNSEEN STUDENT
#          |
#          v
#   first K REAL clips
#          |
#          v
#   personalization
#          |
#          v
#   remaining REAL clips
#
#
# RANDOM protocol:
#
#   K real clips sampled randomly
#
#
# IMPORTANT:
#
#   NO artificial balancing during TESTING.
#   NO synthetic minority examples.
#   NO fabricated support labels.
#
# Training may use balanced episodic sampling.
# Testing MUST use the real support/query protocol.
#
# ============================================================


import json
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

LABELS = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

NUM_CLASSES = 4
INPUT_DIM = 1024

FEATURE_COLS = [
    f"emb_{i}"
    for i in range(INPUT_DIM)
]


# ============================================================
# MODEL DIMENSIONS
# ============================================================

EMBED_DIM = 128

CONTEXT_DIM = 128

LABEL_EMBED_DIM = 32

ATTENTION_HIDDEN = 128

HYPER_HIDDEN = 128


# ============================================================
# TRAINING
# ============================================================

DEFAULT_EPISODES = 2000

DEFAULT_SEED = 42

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 5.0

TRAIN_K_VALUES = [
    5,
    10,
    20,
]

QUERY_MAX = 64


# ============================================================
# H9 LOSS WEIGHTS
# ============================================================

# Main hierarchical objective.
LAMBDA_HIERARCHICAL = 1.0

# Prototype-based objective.
LAMBDA_PROTOTYPE = 0.50

# Supervised contrastive objective.
LAMBDA_SUPCON = 0.10

# Ordinal objective.
LAMBDA_ORDINAL = 0.10


# ============================================================
# SUPERVISED CONTRASTIVE TEMPERATURE
# ============================================================

SUPCON_TEMPERATURE = 0.10


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA LOADING
# ============================================================

def parse_group_from_clip_id(clip_id):

    """
    Extract student/group identifier.

    Expected structure:

        videoX_clipindex_person...

    Example:

        video123_17_person4

    becomes:

        video123_person4
    """

    match = re.match(
        r"^(video[^_]+)_(\d+)_((?:person).+)$",
        str(clip_id)
    )

    if not match:

        return None

    return (
        f"{match.group(1)}_"
        f"{match.group(3)}"
    )


def load_clip_dataframe(json_path):

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    records = []

    for clip_id, meta in data.items():

        if not isinstance(meta, dict):

            continue

        label = meta.get(
            "label"
        )

        if label not in LABELS:

            continue

        embeds = meta.get(
            "embeds"
        )

        if not isinstance(
            embeds,
            list
        ):

            continue

        if len(embeds) != INPUT_DIM:

            continue

        group = parse_group_from_clip_id(
            clip_id
        )

        if group is None:

            continue

        arr = np.asarray(
            embeds,
            dtype=np.float32
        )

        if not np.all(
            np.isfinite(arr)
        ):

            continue

        clip_index_match = re.search(
            r"_(\d+)_",
            str(clip_id)
        )

        clip_index = (
            int(
                clip_index_match.group(1)
            )
            if clip_index_match
            else 0
        )

        records.append(
            {
                "clip_id":
                    str(clip_id),

                "group":
                    group,

                "label":
                    str(label),

                "label_id":
                    LABEL_TO_ID[
                        str(label)
                    ],

                "clip_index":
                    clip_index,

                "embedding":
                    arr,
            }
        )

    if not records:

        raise ValueError(
            "No valid labelled clips found "
            f"in {json_path}"
        )

    df = pd.DataFrame(
        records
    )

    emb_matrix = np.vstack(
        df[
            "embedding"
        ].to_numpy()
    )

    emb_df = pd.DataFrame(
        emb_matrix,
        columns=FEATURE_COLS,
        index=df.index,
    )

    df = pd.concat(
        [
            df.drop(
                columns=[
                    "embedding"
                ]
            ),
            emb_df,
        ],
        axis=1,
    )

    return df


# ============================================================
# FOLD LOADING
# ============================================================

def load_group_fold_map(
    fold_csv_path
):

    fold_df = pd.read_csv(
        fold_csv_path
    )

    required = {
        "group",
        "fold",
    }

    missing = (
        required
        -
        set(
            fold_df.columns
        )
    )

    if missing:

        raise ValueError(
            "Fold CSV missing required "
            f"columns: {sorted(missing)}"
        )

    group_fold = (
        fold_df[
            [
                "group",
                "fold"
            ]
        ]
        .drop_duplicates()
    )

    check = (
        group_fold
        .groupby(
            "group"
        )["fold"]
        .nunique()
    )

    bad = check[
        check != 1
    ]

    if len(bad) > 0:

        raise ValueError(
            "Groups assigned to multiple "
            f"folds: {bad.index.tolist()}"
        )

    return dict(
        zip(
            group_fold[
                "group"
            ],
            group_fold[
                "fold"
            ].astype(int),
        )
    )


# ============================================================
# TEST SUPPORT SAMPLING
# ============================================================

def sample_support_query(
    student_df,
    k_shot,
    protocol,
    rng,
):

    """
    STRICT TEST PROTOCOL.

    early:
        first K REAL clips

    random:
        K REAL clips randomly sampled

    query:
        all remaining REAL clips

    NO BALANCING.
    NO SYNTHETIC EXAMPLES.
    """

    ordered = (
        student_df
        .sort_values(
            "clip_index"
        )
        .copy()
    )

    if len(ordered) <= k_shot:

        return (
            None,
            None,
            (
                f"insufficient_clips_n="
                f"{len(ordered)}_k={k_shot}"
            ),
        )

    if protocol == "early":

        support = (
            ordered.iloc[
                :k_shot
            ]
            .copy()
        )

    elif protocol == "random":

        sampled_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

        support = (
            ordered.loc[
                sampled_idx
            ]
            .copy()
        )

    else:

        return (
            None,
            None,
            f"unknown_protocol_{protocol}",
        )

    query = (
        ordered.loc[
            ~ordered.index.isin(
                support.index
            )
        ]
        .copy()
    )

    if query.empty:

        return (
            None,
            None,
            "empty_query",
        )

    return (
        support,
        query,
        None,
    )


# ============================================================
# TRAINING SUPPORT SAMPLER
# ============================================================

def sample_balanced_support(
    student_df,
    k_shot,
    rng,
):

    """
    TRAINING ONLY.

    Approximately balanced support over classes
    available for this student.

    No synthetic examples.
    """

    available_classes = []

    class_indices = {}

    for class_id in range(
        NUM_CLASSES
    ):

        idx = (
            student_df[
                student_df[
                    "label_id"
                ]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if not available_classes:

        return None

    n_classes = len(
        available_classes
    )

    base = (
        k_shot
        //
        n_classes
    )

    remainder = (
        k_shot
        %
        n_classes
    )

    allocation = {
        c: base
        for c in available_classes
    }

    shuffled = (
        available_classes.copy()
    )

    rng.shuffle(
        shuffled
    )

    for c in shuffled[
        :remainder
    ]:

        allocation[c] += 1

    selected = []

    unused_slots = 0

    for c in available_classes:

        requested = allocation[c]

        available = class_indices[c]

        take = min(
            requested,
            len(available)
        )

        if take < requested:

            unused_slots += (
                requested
                - take
            )

        chosen = rng.choice(
            available,
            size=take,
            replace=False,
        )

        selected.extend(
            chosen.tolist()
        )

    # --------------------------------------------------------
    # Redistribute unused slots.
    # --------------------------------------------------------

    if unused_slots > 0:

        selected_set = set(
            selected
        )

        remaining = []

        for c in available_classes:

            for idx in class_indices[c]:

                if idx not in selected_set:

                    remaining.append(
                        idx
                    )

        if remaining:

            take = min(
                unused_slots,
                len(remaining)
            )

            extra = rng.choice(
                np.asarray(
                    remaining
                ),
                size=take,
                replace=False,
            )

            selected.extend(
                extra.tolist()
            )

    if not selected:

        return None

    return (
        student_df.loc[
            selected
        ]
        .copy()
    )


# ============================================================
# TRAINING QUERY SAMPLER
# ============================================================

def sample_balanced_query(
    query_df,
    max_samples,
    rng,
):

    """
    TRAINING ONLY.

    Approximately class-balanced query batch.
    """

    if query_df.empty:

        return query_df

    available_classes = []

    class_indices = {}

    for class_id in range(
        NUM_CLASSES
    ):

        idx = (
            query_df[
                query_df[
                    "label_id"
                ]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if not available_classes:

        return query_df.iloc[
            0:0
        ]

    n_classes = len(
        available_classes
    )

    per_class = (
        max_samples
        //
        n_classes
    )

    selected = []

    for c in available_classes:

        idx = class_indices[c]

        take = min(
            per_class,
            len(idx)
        )

        if take > 0:

            chosen = rng.choice(
                idx,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # Fill remaining slots.
    selected_set = set(
        selected
    )

    remaining = []

    for c in available_classes:

        for idx in class_indices[c]:

            if idx not in selected_set:

                remaining.append(
                    idx
                )

    slots = (
        max_samples
        - len(selected)
    )

    if (
        slots > 0
        and remaining
    ):

        take = min(
            slots,
            len(remaining)
        )

        extra = rng.choice(
            np.asarray(
                remaining
            ),
            size=take,
            replace=False,
        )

        selected.extend(
            extra.tolist()
        )

    return (
        query_df.loc[
            selected
        ]
        .copy()
    )


# ============================================================
# LABEL-AWARE SUPPORT ENCODER
# ============================================================

class LabelAwareContextEncoder(
    nn.Module
):

    """
    Encodes support clips using:

        visual embedding
             +
        explicit support label

    Then produces:

        global student context
    """

    def __init__(self):

        super().__init__()

        self.embedding_encoder = nn.Sequential(

            nn.Linear(
                INPUT_DIM,
                256
            ),

            nn.ReLU(),

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                EMBED_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                EMBED_DIM
            ),
        )

        self.label_embedding = (
            nn.Embedding(
                NUM_CLASSES,
                LABEL_EMBED_DIM
            )
        )

        self.support_fusion = nn.Sequential(

            nn.Linear(
                EMBED_DIM
                + LABEL_EMBED_DIM,
                CONTEXT_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                CONTEXT_DIM
            ),
        )

        self.attention = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                ATTENTION_HIDDEN
            ),

            nn.Tanh(),

            nn.Linear(
                ATTENTION_HIDDEN,
                1
            ),
        )

        self.context_projection = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                CONTEXT_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                CONTEXT_DIM
            ),
        )

    def forward(
        self,
        support_x,
        support_y,
    ):

        visual = (
            self.embedding_encoder(
                support_x
            )
        )

        labels = (
            self.label_embedding(
                support_y
            )
        )

        fused = torch.cat(
            [
                visual,
                labels,
            ],
            dim=1,
        )

        support_repr = (
            self.support_fusion(
                fused
            )
        )

        attention_logits = (
            self.attention(
                support_repr
            )
        )

        attention = F.softmax(
            attention_logits,
            dim=0,
        )

        context = (
            attention
            * support_repr
        ).sum(
            dim=0
        )

        context = (
            self.context_projection(
                context
            )
        )

        return (
            context,
            attention.squeeze(-1),
            support_repr,
        )


# ============================================================
# QUERY ENCODER
# ============================================================

class QueryEncoder(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                INPUT_DIM,
                256
            ),

            nn.ReLU(),

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                EMBED_DIM
            ),

            nn.ReLU(),

            nn.LayerNorm(
                EMBED_DIM
            ),
        )

    def forward(
        self,
        x
    ):

        return self.net(x)


# ============================================================
# PROTOTYPE CONSTRUCTION
# ============================================================

def build_class_prototypes(
    embeddings,
    labels,
):

    """
    Compute one prototype for each class present
    in the support/query set.

    Prototype:

        mean embedding of class examples
    """

    prototypes = {}

    for class_id in range(
        NUM_CLASSES
    ):

        mask = (
            labels
            == class_id
        )

        if mask.any():

            prototype = (
                embeddings[
                    mask
                ]
                .mean(
                    dim=0
                )
            )

            prototype = F.normalize(
                prototype,
                dim=0
            )

            prototypes[
                class_id
            ] = prototype

    return prototypes


# ============================================================
# PROTOTYPE LOGITS
# ============================================================

def prototype_logits(
    query_embeddings,
    prototypes,
):

    """
    Cosine similarity to available class prototypes.

    Missing support classes receive a very negative score.
    """

    batch_size = (
        query_embeddings.shape[0]
    )

    logits = torch.full(
        (
            batch_size,
            NUM_CLASSES,
        ),
        -10.0,
        device=query_embeddings.device,
    )

    q = F.normalize(
        query_embeddings,
        dim=1
    )

    for class_id, prototype in (
        prototypes.items()
    ):

        logits[
            :,
            class_id
        ] = (
            q
            @
            prototype
        )

    return logits


# ============================================================
# HYPERNETWORK
# ============================================================

class HyperNetwork(
    nn.Module
):

    """
    Generates a personalized 4-class classifier.

    The classifier is used as the main representation
    head alongside the hierarchical branch heads.
    """

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                HYPER_HIDDEN
            ),

            nn.ReLU(),

            nn.LayerNorm(
                HYPER_HIDDEN
            ),

            nn.Linear(
                HYPER_HIDDEN,
                HYPER_HIDDEN
            ),

            nn.ReLU(),
        )

        # Four-class linear head.
        self.output = nn.Linear(
            HYPER_HIDDEN,
            (
                NUM_CLASSES
                * EMBED_DIM
                + NUM_CLASSES
            ),
        )

    def forward(
        self,
        context,
    ):

        params = self.output(
            self.net(
                context
            )
        )

        weight_end = (
            NUM_CLASSES
            * EMBED_DIM
        )

        weight = (
            params[
                :weight_end
            ]
            .view(
                NUM_CLASSES,
                EMBED_DIM
            )
        )

        bias = params[
            weight_end:
        ]

        return (
            weight,
            bias,
        )


# ============================================================
# HIERARCHICAL CLASSIFIER
# ============================================================

class HierarchicalClassifier(
    nn.Module
):

    """
    H9 hierarchical classifier.

    Stage 1:
        disengaged vs engaged

    Stage 2A:
        HD vs D

    Stage 2B:
        E vs HE

    Each branch is personalized from the HyperNetwork
    context.
    """

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Stage 1:
        #
        # 0 = disengaged
        # 1 = engaged
        # ----------------------------------------------------

        self.stage1 = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                64
            ),

            nn.ReLU(),

            nn.LayerNorm(
                64
            ),

            nn.Linear(
                64,
                2
            ),
        )

        # ----------------------------------------------------
        # Disengaged branch:
        #
        # 0 = Highly Disengage
        # 1 = Disengage
        # ----------------------------------------------------

        self.disengaged_branch = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                64
            ),

            nn.ReLU(),

            nn.LayerNorm(
                64
            ),

            nn.Linear(
                64,
                2
            ),
        )

        # ----------------------------------------------------
        # Engaged branch:
        #
        # 0 = Engage
        # 1 = Highly Engage
        # ----------------------------------------------------

        self.engaged_branch = nn.Sequential(

            nn.Linear(
                CONTEXT_DIM,
                64
            ),

            nn.ReLU(),

            nn.LayerNorm(
                64
            ),

            nn.Linear(
                64,
                2
            ),
        )

    def forward(
        self,
        context,
        query_embeddings,
    ):

        batch_size = (
            query_embeddings.shape[0]
        )

        # ----------------------------------------------------
        # Stage 1
        # ----------------------------------------------------

        stage1_context = (
            context
            .unsqueeze(0)
            .expand(
                batch_size,
                -1
            )
        )

        stage1_logits = (
            self.stage1(
                stage1_context
            )
        )

        # ----------------------------------------------------
        # Stage 2
        # ----------------------------------------------------

        disengaged_logits = (
            self.disengaged_branch(
                stage1_context
            )
        )

        engaged_logits = (
            self.engaged_branch(
                stage1_context
            )
        )

        # ----------------------------------------------------
        # Convert hierarchy to four-class probabilities.
        #
        # Class mapping:
        #
        # 0 = HD
        # 1 = D
        # 2 = E
        # 3 = HE
        #
        # Stage 1:
        #
        # index 0 = disengaged
        # index 1 = engaged
        # ----------------------------------------------------

        p_stage1 = F.softmax(
            stage1_logits,
            dim=1
        )

        p_disengaged = (
            p_stage1[
                :,
                0:1
            ]
        )

        p_engaged = (
            p_stage1[
                :,
                1:2
            ]
        )

        p_d_branch = F.softmax(
            disengaged_logits,
            dim=1
        )

        p_e_branch = F.softmax(
            engaged_logits,
            dim=1
        )

        p_hd = (
            p_disengaged
            *
            p_d_branch[
                :,
                0:1
            ]
        )

        p_d = (
            p_disengaged
            *
            p_d_branch[
                :,
                1:2
            ]
        )

        p_e = (
            p_engaged
            *
            p_e_branch[
                :,
                0:1
            ]
        )

        p_he = (
            p_engaged
            *
            p_e_branch[
                :,
                1:2
            ]
        )

        probs = torch.cat(
            [
                p_hd,
                p_d,
                p_e,
                p_he,
            ],
            dim=1,
        )

        # Numerical safety.
        probs = probs.clamp_min(
            1e-8
        )

        hierarchical_logits = (
            torch.log(
                probs
            )
        )

        return {
            "stage1_logits":
                stage1_logits,

            "disengaged_logits":
                disengaged_logits,

            "engaged_logits":
                engaged_logits,

            "hierarchical_logits":
                hierarchical_logits,

            "hierarchical_probs":
                probs,
        }


# ============================================================
# FULL H9 MODEL
# ============================================================

class H9Model(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.context_encoder = (
            LabelAwareContextEncoder()
        )

        self.query_encoder = (
            QueryEncoder()
        )

        self.hypernetwork = (
            HyperNetwork()
        )

        self.hierarchical_classifier = (
            HierarchicalClassifier()
        )

    # --------------------------------------------------------
    # Personalization
    # --------------------------------------------------------

    def personalize(
        self,
        support_x,
        support_y,
    ):

        (
            context,
            attention,
            support_repr,
        ) = self.context_encoder(
            support_x,
            support_y,
        )

        (
            weight,
            bias,
        ) = self.hypernetwork(
            context
        )

        return (
            context,
            attention,
            support_repr,
            weight,
            bias,
        )

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    def forward(
        self,
        support_x,
        support_y,
        query_x,
    ):

        (
            context,
            attention,
            support_repr,
            weight,
            bias,
        ) = self.personalize(
            support_x,
            support_y,
        )

        query_embeddings = (
            self.query_encoder(
                query_x
            )
        )

        # ----------------------------------------------------
        # HyperNetwork 4-class logits
        # ----------------------------------------------------

        hyper_logits = F.linear(
            query_embeddings,
            weight,
            bias,
        )

        # ----------------------------------------------------
        # Hierarchical logits
        # ----------------------------------------------------

        hierarchy = (
            self.hierarchical_classifier(
                context,
                query_embeddings,
            )
        )

        # ----------------------------------------------------
        # Support prototypes
        # ----------------------------------------------------

        support_embeddings = (
            self.query_encoder(
                support_x
            )
        )

        prototypes = (
            build_class_prototypes(
                support_embeddings,
                support_y,
            )
        )

        proto_logits = (
            prototype_logits(
                query_embeddings,
                prototypes,
            )
        )

        return {
            "query_embeddings":
                query_embeddings,

            "support_embeddings":
                support_embeddings,

            "context":
                context,

            "attention":
                attention,

            "hyper_logits":
                hyper_logits,

            "proto_logits":
                proto_logits,

            "hierarchy":
                hierarchy,

            "prototypes":
                prototypes,
        }


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

def supervised_contrastive_loss(
    embeddings,
    labels,
    temperature=SUPCON_TEMPERATURE,
):

    """
    Supervised contrastive loss.

    Same-label examples:
        pulled together

    Different-label examples:
        pushed apart

    If a batch contains no positive pair for a sample,
    that sample is ignored.

    This is especially useful with the class-balanced
    training query sampler.
    """

    if embeddings.shape[0] < 2:

        return embeddings.new_tensor(
            0.0
        )

    z = F.normalize(
        embeddings,
        dim=1
    )

    similarity = (
        torch.matmul(
            z,
            z.T
        )
        /
        temperature
    )

    batch_size = (
        embeddings.shape[0]
    )

    logits_mask = (
        torch.ones_like(
            similarity
        )
        -
        torch.eye(
            batch_size,
            device=embeddings.device
        )
    )

    similarity = (
        similarity
        -
        similarity.max(
            dim=1,
            keepdim=True
        ).values.detach()
    )

    exp_sim = (
        torch.exp(
            similarity
        )
        *
        logits_mask
    )

    labels = labels.view(
        -1,
        1
    )

    positive_mask = (
        torch.eq(
            labels,
            labels.T
        )
        .float()
        *
        logits_mask
    )

    denominator = (
        exp_sim.sum(
            dim=1,
            keepdim=True
        )
        +
        1e-8
    )

    log_prob = (
        similarity
        -
        torch.log(
            denominator
        )
    )

    positive_count = (
        positive_mask.sum(
            dim=1
        )
    )

    mean_log_prob_pos = (
        (
            positive_mask
            *
            log_prob
        )
        .sum(
            dim=1
        )
        /
        positive_count.clamp_min(
            1.0
        )
    )

    valid = (
        positive_count
        > 0
    )

    if not valid.any():

        return embeddings.new_tensor(
            0.0
        )

    loss = (
        -mean_log_prob_pos[
            valid
        ]
        .mean()
    )

    return loss


# ============================================================
# ORDINAL REGULARIZATION
# ============================================================

def ordinal_loss(
    logits,
    targets,
):

    """
    CORAL-style ordinal consistency.

    Class order:

        0 = Highly Disengage
        1 = Disengage
        2 = Engage
        3 = Highly Engage

    Construct three binary cumulative tasks:

        y > 0
        y > 1
        y > 2

    The corresponding logits are trained to be ordered.

    This does NOT replace the four-class objective.
    It is a small regularizer.
    """

    # --------------------------------------------------------
    # Convert four-class logits to probabilities.
    # --------------------------------------------------------

    probs = F.softmax(
        logits,
        dim=1
    )

    cumulative_probs = torch.stack(
        [
            probs[
                :,
                1
            ]
            + probs[
                :,
                2
            ]
            + probs[
                :,
                3
            ],

            probs[
                :,
                2
            ]
            + probs[
                :,
                3
            ],

            probs[
                :,
                3
            ],
        ],
        dim=1,
    )

    cumulative_targets = torch.stack(
        [
            (
                targets > 0
            ).float(),

            (
                targets > 1
            ).float(),

            (
                targets > 2
            ).float(),
        ],
        dim=1,
    )

    loss = F.binary_cross_entropy(
        cumulative_probs.clamp(
            1e-6,
            1.0 - 1e-6
        ),
        cumulative_targets,
    )

    # --------------------------------------------------------
    # Encourage monotonic cumulative probabilities:
    #
    # P(y>0) >= P(y>1) >= P(y>2)
    # --------------------------------------------------------

    monotonic_penalty = (
        F.relu(
            cumulative_probs[
                :,
                1
            ]
            -
            cumulative_probs[
                :,
                0
            ]
        ).mean()
        +
        F.relu(
            cumulative_probs[
                :,
                2
            ]
            -
            cumulative_probs[
                :,
                1
            ]
        ).mean()
    )

    return (
        loss
        +
        0.10
        * monotonic_penalty
    )


# ============================================================
# HIERARCHICAL LOSS
# ============================================================

def hierarchical_loss(
    hierarchy,
    targets,
):

    """
    Explicitly trains the hierarchy.

    Stage 1:
        HD/D -> disengaged
        E/HE -> engaged

    Stage 2A:
        HD vs D

    Stage 2B:
        E vs HE

    Only the appropriate second-stage branch receives
    the second-stage loss.
    """

    stage1_logits = (
        hierarchy[
            "stage1_logits"
        ]
    )

    disengaged_logits = (
        hierarchy[
            "disengaged_logits"
        ]
    )

    engaged_logits = (
        hierarchy[
            "engaged_logits"
        ]
    )

    # --------------------------------------------------------
    # Stage 1 targets.
    #
    # 0 = disengaged
    # 1 = engaged
    # --------------------------------------------------------

    stage1_targets = torch.where(
        targets <= 1,
        torch.zeros_like(
            targets
        ),
        torch.ones_like(
            targets
        ),
    )

    loss_stage1 = F.cross_entropy(
        stage1_logits,
        stage1_targets,
    )

    # --------------------------------------------------------
    # Disengaged branch.
    #
    # HD = 0
    # D  = 1
    # --------------------------------------------------------

    disengaged_mask = (
        targets <= 1
    )

    if disengaged_mask.any():

        disengaged_targets = (
            targets[
                disengaged_mask
            ]
        )

        loss_disengaged = (
            F.cross_entropy(
                disengaged_logits[
                    disengaged_mask
                ],
                disengaged_targets,
            )
        )

    else:

        loss_disengaged = (
            targets.new_tensor(
                0.0,
                dtype=torch.float32
            )
        )

    # --------------------------------------------------------
    # Engaged branch.
    #
    # E  = 0
    # HE = 1
    # --------------------------------------------------------

    engaged_mask = (
        targets >= 2
    )

    if engaged_mask.any():

        engaged_targets = (
            targets[
                engaged_mask
            ]
            - 2
        )

        loss_engaged = (
            F.cross_entropy(
                engaged_logits[
                    engaged_mask
                ],
                engaged_targets,
            )
        )

    else:

        loss_engaged = (
            targets.new_tensor(
                0.0,
                dtype=torch.float32
            )
        )

    return (
        loss_stage1
        +
        loss_disengaged
        +
        loss_engaged
    )


# ============================================================
# COMBINED H9 LOSS
# ============================================================

def h9_loss(
    outputs,
    targets,
):

    # --------------------------------------------------------
    # Main hierarchy
    # --------------------------------------------------------

    loss_hier = (
        hierarchical_loss(
            outputs[
                "hierarchy"
            ],
            targets,
        )
    )

    # --------------------------------------------------------
    # HyperNetwork four-class classifier
    # --------------------------------------------------------

    loss_hyper = F.cross_entropy(
        outputs[
            "hyper_logits"
        ],
        targets,
    )

    # --------------------------------------------------------
    # Prototype classifier.
    #
    # Only include this when at least one prototype exists.
    # --------------------------------------------------------

    proto_logits = outputs[
        "proto_logits"
    ]

    loss_proto = F.cross_entropy(
        proto_logits,
        targets,
    )

    # --------------------------------------------------------
    # Supervised contrastive loss
    # --------------------------------------------------------

    loss_supcon = (
        supervised_contrastive_loss(
            outputs[
                "query_embeddings"
            ],
            targets,
        )
    )

    # --------------------------------------------------------
    # Ordinal loss
    # --------------------------------------------------------

    loss_ord = (
        ordinal_loss(
            outputs[
                "hyper_logits"
            ],
            targets,
        )
    )

    # --------------------------------------------------------
    # Total
    # --------------------------------------------------------

    total = (

        LAMBDA_HIERARCHICAL
        * loss_hier

        +

        0.50
        * loss_hyper

        +

        LAMBDA_PROTOTYPE
        * loss_proto

        +

        LAMBDA_SUPCON
        * loss_supcon

        +

        LAMBDA_ORDINAL
        * loss_ord
    )

    components = {

        "total":
            total.item(),

        "hierarchical":
            loss_hier.item(),

        "hyper":
            loss_hyper.item(),

        "prototype":
            loss_proto.item(),

        "supcon":
            loss_supcon.item(),

        "ordinal":
            loss_ord.item(),
    }

    return (
        total,
        components,
    )


# ============================================================
# EPISODIC META-TRAINING
# ============================================================

def episodic_train(
    model,
    train_df,
    scaler,
    episodes,
    seed,
    device,
):

    """
    H9 episodic meta-training.

    TRAINING:
        class-aware support
        class-aware query

    TEST:
        real support
        real query
    """

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    groups = sorted(
        train_df[
            "group"
        ]
        .unique()
    )

    if not groups:

        raise ValueError(
            "No training students."
        )

    successful = 0

    running_losses = []

    for episode in range(
        episodes
    ):

        # ----------------------------------------------------
        # Select training student.
        # ----------------------------------------------------

        group = rng.choice(
            groups
        )

        student_df = (
            train_df[
                train_df[
                    "group"
                ]
                == group
            ]
            .copy()
        )

        # ----------------------------------------------------
        # Select K.
        # ----------------------------------------------------

        k = int(
            rng.choice(
                TRAIN_K_VALUES
            )
        )

        if len(student_df) <= k:

            continue

        # ----------------------------------------------------
        # Class-aware support.
        # ----------------------------------------------------

        support_df = (
            sample_balanced_support(
                student_df,
                k,
                rng,
            )
        )

        if (
            support_df is None
            or support_df.empty
        ):

            continue

        # ----------------------------------------------------
        # Remaining query.
        # ----------------------------------------------------

        query_df = (
            student_df.loc[
                ~student_df.index.isin(
                    support_df.index
                )
            ]
            .copy()
        )

        if query_df.empty:

            continue

        # ----------------------------------------------------
        # Class-aware query.
        # ----------------------------------------------------

        query_df = (
            sample_balanced_query(
                query_df,
                QUERY_MAX,
                rng,
            )
        )

        if query_df.empty:

            continue

        # ----------------------------------------------------
        # Transform using training-only scaler.
        # ----------------------------------------------------

        support_x_np = scaler.transform(
            support_df[
                FEATURE_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        query_x_np = scaler.transform(
            query_df[
                FEATURE_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        support_y_np = (
            support_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        query_y_np = (
            query_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        support_x = torch.tensor(
            support_x_np,
            dtype=torch.float32,
            device=device,
        )

        support_y = torch.tensor(
            support_y_np,
            dtype=torch.long,
            device=device,
        )

        query_x = torch.tensor(
            query_x_np,
            dtype=torch.float32,
            device=device,
        )

        query_y = torch.tensor(
            query_y_np,
            dtype=torch.long,
            device=device,
        )

        # ----------------------------------------------------
        # Forward.
        # ----------------------------------------------------

        outputs = model(
            support_x,
            support_y,
            query_x,
        )

        # ----------------------------------------------------
        # Loss.
        # ----------------------------------------------------

        loss, components = (
            h9_loss(
                outputs,
                query_y,
            )
        )

        # ----------------------------------------------------
        # Backprop.
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP,
        )

        optimizer.step()

        successful += 1

        running_losses.append(
            loss.item()
        )

        # ----------------------------------------------------
        # Progress.
        # ----------------------------------------------------

        if (
            (episode + 1)
            % 100
            == 0
        ):

            recent = (
                np.mean(
                    running_losses[
                        -100:
                    ]
                )
            )

            print(
                f"    Episode "
                f"{episode + 1:4d}/"
                f"{episodes} | "
                f"loss={recent:.4f} | "
                f"hier="
                f"{components['hierarchical']:.4f} | "
                f"proto="
                f"{components['prototype']:.4f} | "
                f"supcon="
                f"{components['supcon']:.4f} | "
                f"ord="
                f"{components['ordinal']:.4f}"
            )

    print(
        f"    Successful episodes: "
        f"{successful}/{episodes}"
    )

    if running_losses:

        print(
            "    Final mean training loss: "
            f"{np.mean(running_losses[-100:]):.4f}"
        )

    return model


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0,
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    return {

        "accuracy":
            float(
                accuracy
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "support":
            support,

        "confusion_matrix":
            cm,
    }


# ============================================================
# MAJORITY BASELINE
# ============================================================

def majority_prediction(
    y_query,
):

    counts = Counter(
        y_query.tolist()
    )

    majority_class = max(
        counts,
        key=counts.get
    )

    return np.full(
        len(y_query),
        majority_class,
        dtype=np.int64,
    )


# ============================================================
# HIERARCHICAL PREDICTION
# ============================================================

def hierarchical_predict(
    hierarchy,
):

    """
    Convert hierarchical probability output into
    the final four-class prediction.

    Since hierarchical_probs contains:

        P(HD)
        P(D)
        P(E)
        P(HE)

    prediction is simply argmax.
    """

    probs = hierarchy[
        "hierarchical_probs"
    ]

    return (
        torch.argmax(
            probs,
            dim=1
        )
    )


# ============================================================
# EVALUATION
# ============================================================

def evaluate_student(
    model,
    support_df,
    query_df,
    scaler,
    device,
):

    # --------------------------------------------------------
    # Transform.
    # --------------------------------------------------------

    support_x_np = scaler.transform(
        support_df[
            FEATURE_COLS
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    query_x_np = scaler.transform(
        query_df[
            FEATURE_COLS
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    support_y_np = (
        support_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    y_true = (
        query_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    support_x = torch.tensor(
        support_x_np,
        dtype=torch.float32,
        device=device,
    )

    support_y = torch.tensor(
        support_y_np,
        dtype=torch.long,
        device=device,
    )

    query_x = torch.tensor(
        query_x_np,
        dtype=torch.float32,
        device=device,
    )

    model.eval()

    with torch.no_grad():

        outputs = model(
            support_x,
            support_y,
            query_x,
        )

        # ----------------------------------------------------
        # H9 hierarchical prediction.
        # ----------------------------------------------------

        y_pred_hier = (
            hierarchical_predict(
                outputs[
                    "hierarchy"
                ]
            )
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # HyperNetwork prediction.
        # ----------------------------------------------------

        y_pred_hyper = (
            torch.argmax(
                outputs[
                    "hyper_logits"
                ],
                dim=1,
            )
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Prototype prediction.
        # ----------------------------------------------------

        y_pred_proto = (
            torch.argmax(
                outputs[
                    "proto_logits"
                ],
                dim=1,
            )
            .cpu()
            .numpy()
        )

    # --------------------------------------------------------
    # Metrics for the H9 hierarchical prediction.
    # --------------------------------------------------------

    metrics = calculate_metrics(
        y_true,
        y_pred_hier,
    )

    # --------------------------------------------------------
    # Metrics for HyperNetwork head.
    # --------------------------------------------------------

    hyper_metrics = calculate_metrics(
        y_true,
        y_pred_hyper,
    )

    # --------------------------------------------------------
    # Metrics for prototype head.
    # --------------------------------------------------------

    proto_metrics = calculate_metrics(
        y_true,
        y_pred_proto,
    )

    # --------------------------------------------------------
    # Majority baseline.
    # --------------------------------------------------------

    majority_pred = (
        majority_prediction(
            y_true
        )
    )

    majority_metrics = (
        calculate_metrics(
            y_true,
            majority_pred,
        )
    )

    return {

        "y_true":
            y_true,

        "y_pred":
            y_pred_hier,

        "hyper_pred":
            y_pred_hyper,

        "proto_pred":
            y_pred_proto,

        "metrics":
            metrics,

        "hyper_metrics":
            hyper_metrics,

        "proto_metrics":
            proto_metrics,

        "majority_metrics":
            majority_metrics,

        "num_prototype_classes":
            len(
                outputs[
                    "prototypes"
                ]
            ),

        "support_labels":
            support_y_np,
    }


# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(
    clip_df,
    group_fold,
    k_values,
    seed,
    episodes,
    device,
    output_dir,
):

    eligible_groups = set(
        group_fold.keys()
    )

    eval_df = clip_df[
        clip_df[
            "group"
        ].isin(
            eligible_groups
        )
    ].copy()

    summary_rows = []

    per_student_rows = []

    per_class_rows = []

    confusion_rows = []

    skipped_rows = []

    # --------------------------------------------------------
    # Outer folds.
    # --------------------------------------------------------

    fold_ids = sorted(
        set(
            group_fold.values()
        )
    )

    for fold_id in fold_ids:

        print()
        print(
            "=" * 80
        )

        print(
            f"H9 OUTER FOLD {fold_id}"
        )

        print(
            "=" * 80
        )

        test_groups = [
            g
            for g, f
            in group_fold.items()
            if f == fold_id
        ]

        train_groups = [
            g
            for g, f
            in group_fold.items()
            if f != fold_id
        ]

        train_df = eval_df[
            eval_df[
                "group"
            ].isin(
                train_groups
            )
        ].copy()

        test_df = eval_df[
            eval_df[
                "group"
            ].isin(
                test_groups
            )
        ].copy()

        if train_df.empty:

            raise ValueError(
                f"No training data "
                f"for fold {fold_id}"
            )

        # ----------------------------------------------------
        # Leakage check.
        # ----------------------------------------------------

        overlap = (
            set(
                train_df[
                    "group"
                ]
            )
            &
            set(
                test_df[
                    "group"
                ]
            )
        )

        if overlap:

            raise RuntimeError(
                "DATA LEAKAGE: "
                f"{overlap}"
            )

        print(
            "Training students:",
            train_df[
                "group"
            ].nunique()
        )

        print(
            "Test students:",
            test_df[
                "group"
            ].nunique()
        )

        print()
        print(
            "Training class distribution:"
        )

        print(
            train_df[
                "label"
            ]
            .value_counts()
        )

        # ----------------------------------------------------
        # Training-only scaler.
        # ----------------------------------------------------

        scaler = StandardScaler()

        scaler.fit(
            train_df[
                FEATURE_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        # ----------------------------------------------------
        # Fresh H9 model.
        # ----------------------------------------------------

        set_seed(
            seed
            + fold_id
        )

        model = H9Model()

        model.to(
            device
        )

        print()
        print(
            f"H9 meta-training "
            f"for {episodes} episodes..."
        )

        episodic_train(
            model=model,
            train_df=train_df,
            scaler=scaler,
            episodes=episodes,
            seed=(
                seed
                +
                fold_id * 1000
            ),
            device=device,
        )

        # ----------------------------------------------------
        # Evaluate every unseen student.
        # ----------------------------------------------------

        for group in test_groups:

            student_df = (
                test_df[
                    test_df[
                        "group"
                    ]
                    == group
                ]
                .copy()
            )

            if student_df.empty:

                continue

            for k in k_values:

                for protocol in [
                    "early",
                    "random",
                ]:

                    local_seed = (
                        seed
                        +
                        fold_id * 10000
                        +
                        k * 100
                        +
                        sum(
                            ord(ch)
                            for ch
                            in group
                        )
                        +
                        (
                            1
                            if protocol
                            == "early"
                            else 0
                        )
                    )

                    rng = (
                        np.random.default_rng(
                            local_seed
                        )
                    )

                    (
                        support_df,
                        query_df,
                        reason,
                    ) = (
                        sample_support_query(
                            student_df,
                            k,
                            protocol,
                            rng,
                        )
                    )

                    if reason is not None:

                        skipped_rows.append(
                            {

                                "method":
                                    "H9_hierarchical",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "reason":
                                    reason,
                            }
                        )

                        continue

                    # ------------------------------------------------
                    # Evaluate.
                    # ------------------------------------------------

                    result = (
                        evaluate_student(
                            model=model,
                            support_df=support_df,
                            query_df=query_df,
                            scaler=scaler,
                            device=device,
                        )
                    )

                    metrics = (
                        result[
                            "metrics"
                        ]
                    )

                    hyper_metrics = (
                        result[
                            "hyper_metrics"
                        ]
                    )

                    proto_metrics = (
                        result[
                            "proto_metrics"
                        ]
                    )

                    majority_metrics = (
                        result[
                            "majority_metrics"
                        ]
                    )

                    y_true = (
                        result[
                            "y_true"
                        ]
                    )

                    y_pred = (
                        result[
                            "y_pred"
                        ]
                    )

                    # ------------------------------------------------
                    # Student-level row.
                    # ------------------------------------------------

                    per_student_rows.append(
                        {

                            "method":
                                "H9_hierarchical",

                            "outer_fold":
                                fold_id,

                            "student_group":
                                group,

                            "k_shot":
                                k,

                            "support_protocol":
                                protocol,

                            "support_size":
                                len(
                                    support_df
                                ),

                            "query_size":
                                len(
                                    query_df
                                ),

                            "prototype_classes_available":
                                result[
                                    "num_prototype_classes"
                                ],

                            "accuracy":
                                metrics[
                                    "accuracy"
                                ],

                            "balanced_accuracy":
                                metrics[
                                    "balanced_accuracy"
                                ],

                            "macro_f1":
                                metrics[
                                    "macro_f1"
                                ],

                            "hyper_accuracy":
                                hyper_metrics[
                                    "accuracy"
                                ],

                            "hyper_balanced_accuracy":
                                hyper_metrics[
                                    "balanced_accuracy"
                                ],

                            "hyper_macro_f1":
                                hyper_metrics[
                                    "macro_f1"
                                ],

                            "prototype_accuracy":
                                proto_metrics[
                                    "accuracy"
                                ],

                            "prototype_balanced_accuracy":
                                proto_metrics[
                                    "balanced_accuracy"
                                ],

                            "prototype_macro_f1":
                                proto_metrics[
                                    "macro_f1"
                                ],

                            "majority_accuracy":
                                majority_metrics[
                                    "accuracy"
                                ],

                            "majority_balanced_accuracy":
                                majority_metrics[
                                    "balanced_accuracy"
                                ],

                            "majority_macro_f1":
                                majority_metrics[
                                    "macro_f1"
                                ],
                        }
                    )

                    # ------------------------------------------------
                    # Per-class metrics.
                    # ------------------------------------------------

                    for class_id, label in enumerate(
                        LABELS
                    ):

                        per_class_rows.append(
                            {

                                "method":
                                    "H9_hierarchical",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "class":
                                    label,

                                "precision":
                                    float(
                                        metrics[
                                            "precision"
                                        ][
                                            class_id
                                        ]
                                    ),

                                "recall":
                                    float(
                                        metrics[
                                            "recall"
                                        ][
                                            class_id
                                        ]
                                    ),

                                "f1":
                                    float(
                                        metrics[
                                            "f1"
                                        ][
                                            class_id
                                        ]
                                    ),

                                "support":
                                    int(
                                        metrics[
                                            "support"
                                        ][
                                            class_id
                                        ]
                                    ),
                            }
                        )

                    # ------------------------------------------------
                    # Confusion matrix.
                    # ------------------------------------------------

                    cm = metrics[
                        "confusion_matrix"
                    ]

                    for true_id in range(
                        NUM_CLASSES
                    ):

                        for pred_id in range(
                            NUM_CLASSES
                        ):

                            confusion_rows.append(
                                {

                                    "method":
                                        "H9_hierarchical",

                                    "outer_fold":
                                        fold_id,

                                    "student_group":
                                        group,

                                    "k_shot":
                                        k,

                                    "support_protocol":
                                        protocol,

                                    "true_class":
                                        LABELS[
                                            true_id
                                        ],

                                    "predicted_class":
                                        LABELS[
                                            pred_id
                                        ],

                                    "count":
                                        int(
                                            cm[
                                                true_id,
                                                pred_id
                                            ]
                                        ),
                                }
                            )

    # ========================================================
    # DATAFRAMES
    # ========================================================

    per_student_df = pd.DataFrame(
        per_student_rows
    )

    per_class_df = pd.DataFrame(
        per_class_rows
    )

    confusion_df = pd.DataFrame(
        confusion_rows
    )

    skipped_df = pd.DataFrame(
        skipped_rows
    )

    if per_student_df.empty:

        raise RuntimeError(
            "No evaluations produced."
        )

    # ========================================================
    # MAIN SUMMARY
    # ========================================================

    summary_df = (
        per_student_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
            ],
            as_index=False,
        )
        .agg(

            n_students=(
                "student_group",
                "nunique",
            ),

            total_query_clips=(
                "query_size",
                "sum",
            ),

            mean_accuracy=(
                "accuracy",
                "mean",
            ),

            sd_accuracy=(
                "accuracy",
                "std",
            ),

            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),

            sd_balanced_accuracy=(
                "balanced_accuracy",
                "std",
            ),

            mean_macro_f1=(
                "macro_f1",
                "mean",
            ),

            sd_macro_f1=(
                "macro_f1",
                "std",
            ),

            mean_hyper_accuracy=(
                "hyper_accuracy",
                "mean",
            ),

            mean_prototype_accuracy=(
                "prototype_accuracy",
                "mean",
            ),

            mean_prototype_balanced_accuracy=(
                "prototype_balanced_accuracy",
                "mean",
            ),

            mean_majority_accuracy=(
                "majority_accuracy",
                "mean",
            ),

            mean_majority_balanced_accuracy=(
                "majority_balanced_accuracy",
                "mean",
            ),

            mean_majority_macro_f1=(
                "majority_macro_f1",
                "mean",
            ),

            mean_prototype_classes_available=(
                "prototype_classes_available",
                "mean",
            ),
        )
        .sort_values(
            [
                "support_protocol",
                "k_shot",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # PER-CLASS SUMMARY
    # ========================================================

    class_summary = (
        per_class_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
                "class",
            ],
            as_index=False,
        )
        .agg(

            mean_precision=(
                "precision",
                "mean",
            ),

            mean_recall=(
                "recall",
                "mean",
            ),

            mean_f1=(
                "f1",
                "mean",
            ),

            total_support=(
                "support",
                "sum",
            ),
        )
    )

    # ========================================================
    # OUTPUT DIRECTORY
    # ========================================================

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    per_student_path = os.path.join(
        output_dir,
        "h9_per_student.csv",
    )

    summary_path = os.path.join(
        output_dir,
        "h9_summary.csv",
    )

    per_class_path = os.path.join(
        output_dir,
        "h9_per_class.csv",
    )

    confusion_path = os.path.join(
        output_dir,
        "h9_confusion_matrix.csv",
    )

    skipped_path = os.path.join(
        output_dir,
        "h9_skipped.csv",
    )

    per_student_df.to_csv(
        per_student_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    class_summary.to_csv(
        per_class_path,
        index=False,
    )

    confusion_df.to_csv(
        confusion_path,
        index=False,
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print()
    print(
        "=" * 80
    )

    print(
        "H9: HIERARCHICAL "
        "HYPERNETWORK RESULTS"
    )

    print(
        "=" * 80
    )

    print(
        summary_df.to_string(
            index=False
        )
    )

    # ========================================================
    # PER-CLASS
    # ========================================================

    print()
    print(
        "=" * 80
    )

    print(
        "H9 PER-CLASS RESULTS"
    )

    print(
        "=" * 80
    )

    print(
        class_summary.to_string(
            index=False
        )
    )

    # ========================================================
    # HIGHLIGHT HIGHLY DISENGAGE
    # ========================================================

    print()
    print(
        "=" * 80
    )

    print(
        "H9 HIGHLY DISENGAGE RESULTS"
    )

    print(
        "=" * 80
    )

    hd_summary = class_summary[
        class_summary[
            "class"
        ]
        ==
        "Highly Disengage"
    ].copy()

    print(
        hd_summary.to_string(
            index=False
        )
    )

    # ========================================================
    # SKIPPED
    # ========================================================

    print()
    print(
        "=" * 80
    )

    print(
        "H9 SKIPPED CASES"
    )

    print(
        "=" * 80
    )

    if skipped_df.empty:

        print(
            "None."
        )

    else:

        print(
            skipped_df[
                "reason"
            ]
            .value_counts()
        )

    # ========================================================
    # FILES
    # ========================================================

    print()
    print(
        "=" * 80
    )

    print(
        "H9 SAVED FILES"
    )

    print(
        "=" * 80
    )

    print(
        per_student_path
    )

    print(
        summary_path
    )

    print(
        per_class_path
    )

    print(
        confusion_path
    )

    print(
        skipped_path
    )

    return (
        per_student_df,
        summary_df,
        class_summary,
        confusion_df,
        skipped_df,
    )


# ============================================================
# COLAB MAIN
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # PATHS
    # --------------------------------------------------------

    JSON_PATH = (
        "/content/final_data_1.json"
    )

    FOLDS_PATH = (
        "/content/"
        "cmose_h1_h2_fold_protocol.csv"
    )

    OUTPUT_DIR = (
        "/content/h9_results"
    )

    # --------------------------------------------------------
    # EXPERIMENT CONFIG
    # --------------------------------------------------------

    K_VALUES = [
        5,
        10,
        20,
    ]

    # --------------------------------------------------------
    # Start with 2000.
    #
    # If GPU runtime is comfortable, 3000 can be tested
    # after verifying the first run.
    # --------------------------------------------------------

    EPISODES = 2000

    SEED = 42

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------

    if torch.cuda.is_available():

        DEVICE = torch.device(
            "cuda"
        )

        print(
            "CUDA available: True"
        )

        print(
            "GPU:",
            torch.cuda.get_device_name(
                0
            )
        )

    else:

        DEVICE = torch.device(
            "cpu"
        )

        print(
            "CUDA available: False"
        )

        print(
            "Using CPU."
        )

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    set_seed(
        SEED
    )

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    print()
    print(
        "=" * 80
    )

    print(
        "LOADING DATA"
    )

    print(
        "=" * 80
    )

    clip_df = (
        load_clip_dataframe(
            JSON_PATH
        )
    )

    print(
        f"Loaded "
        f"{len(clip_df)} labelled clips "
        f"from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    # --------------------------------------------------------
    # Overall distribution
    # --------------------------------------------------------

    print()
    print(
        "Overall class distribution:"
    )

    print(
        clip_df[
            "label"
        ]
        .value_counts()
    )

    print()
    print(
        "Overall class proportions:"
    )

    print(
        clip_df[
            "label"
        ]
        .value_counts(
            normalize=True
        )
    )

    # --------------------------------------------------------
    # Fold protocol
    # --------------------------------------------------------

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            FOLDS_PATH
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # --------------------------------------------------------
    # Run H9
    # --------------------------------------------------------

    run_experiment(
        clip_df=clip_df,

        group_fold=group_fold,

        k_values=K_VALUES,

        seed=SEED,

        episodes=EPISODES,

        device=DEVICE,

        output_dir=OUTPUT_DIR,
    )

CUDA available: True
GPU: Tesla T4

LOADING DATA
Loaded 11902 labelled clips from 103 students.

Overall class distribution:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Overall class proportions:
label
Engage              0.692741
Disengage           0.182322
Highly Engage       0.096202
Highly Disengage    0.028735
Name: proportion, dtype: float64

Loading fold protocol...
Fold protocol contains 82 students.

H9 OUTER FOLD 1
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6291
Disengage           1601
Highly Engage        904
Highly Disengage     263
Name: count, dtype: int64

H9 meta-training for 2000 episodes...
    Episode  100/2000 | loss=3.4782 | hier=1.4652 | proto=1.3653 | supcon=4.3806 | ord=0.5572
    Episode  200/2000 | loss=2.7383 | hier=2.1346 | proto=1.2038 | supcon=4.0931 | ord=0.2896
    Episode  300/2000 | loss=2.5227 | hier=0.6797 | 

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H9 OUTER FOLD 2
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6415
Disengage           1605
Highly Engage        893
Highly Disengage     259
Name: count, dtype: int64

H9 meta-training for 2000 episodes...
    Episode  100/2000 | loss=3.5803 | hier=1.2367 | proto=1.0224 | supcon=4.3896 | ord=0.4342
    Episode  200/2000 | loss=2.5331 | hier=1.1621 | proto=1.2811 | supcon=4.4450 | ord=0.2583
    Episode  300/2000 | loss=2.5955 | hier=0.8370 | proto=1.0319 | supcon=3.8802 | ord=0.2133
    Episode  400/2000 | loss=2.4835 | hier=0.3061 | proto=0.7431 | supcon=4.0580 | ord=0.0681
    Episode  500/2000 | loss=2.4623 | hier=0.6052 | proto=0.6895 | supcon=3.5183 | ord=0.1302
    Episode  600/2000 | loss=2.3033 | hier=0.4476 | proto=0.6889 | supcon=3.5428 | ord=0.1312
    Episode  700/2000 | loss=2.2729 | hier=0.3927 | proto=1.0155 | supcon=4.4198 | ord=0.0823
    Episode  800/2000 | loss=2.2852 | hier=0.8019 | proto=1.3336 | supcon=3.4173 | o

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H9 OUTER FOLD 3
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6459
Disengage           1628
Highly Engage        872
Highly Disengage     251
Name: count, dtype: int64

H9 meta-training for 2000 episodes...
    Episode  100/2000 | loss=3.3423 | hier=0.7911 | proto=1.0394 | supcon=4.5293 | ord=0.2815
    Episode  200/2000 | loss=2.7736 | hier=1.0041 | proto=1.0433 | supcon=4.1624 | ord=0.2110
    Episode  300/2000 | loss=2.6448 | hier=1.1623 | proto=1.1344 | supcon=4.3341 | ord=0.2700
    Episode  400/2000 | loss=2.5502 | hier=0.9294 | proto=1.0895 | supcon=4.2369 | ord=0.2209
    Episode  500/2000 | loss=2.4407 | hier=1.0511 | proto=1.0116 | supcon=4.5457 | ord=0.2234
    Episode  600/2000 | loss=2.3779 | hier=1.3734 | proto=0.8081 | supcon=4.7735 | ord=0.1593
    Episode  700/2000 | loss=2.4837 | hier=1.3555 | proto=0.9848 | supcon=3.9735 | ord=0.3000
    Episode  800/2000 | loss=2.2185 | hier=0.3452 | proto=1.3297 | supcon=3.9126 | o

...


H9 OUTER FOLD 4
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6111
Disengage           1623
Highly Engage        815
Highly Disengage     237
Name: count, dtype: int64

H9 meta-training for 2000 episodes...
    Episode  100/2000 | loss=3.4819 | hier=0.7582 | proto=0.9475 | supcon=3.5112 | ord=0.1401
    Episode  200/2000 | loss=2.5083 | hier=0.4789 | proto=1.1937 | supcon=4.1664 | ord=0.0398
    Episode  300/2000 | loss=2.5949 | hier=0.9209 | proto=0.8086 | supcon=4.3287 | ord=0.1249
    Episode  400/2000 | loss=2.3270 | hier=0.5610 | proto=1.1798 | supcon=4.3731 | ord=0.0878
    Episode  500/2000 | loss=2.3304 | hier=0.8886 | proto=1.3098 | supcon=4.3398 | ord=0.2321
    Episode  600/2000 | loss=2.3306 | hier=0.8954 | proto=0.9935 | supcon=3.3251 | ord=0.1134
    Episode  700/2000 | loss=2.4647 | hier=1.9850 | proto=1.2032 | supcon=3.9121 | ord=0.2751
    Episode  800/2000 | loss=2.1291 | hier=1.0863 | proto=1.3402 | supcon=4.1109 | o

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H9 OUTER FOLD 5
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6568
Disengage           1727
Highly Engage        912
Highly Disengage     270
Name: count, dtype: int64

H9 meta-training for 2000 episodes...
    Episode  100/2000 | loss=3.4872 | hier=0.6912 | proto=0.8658 | supcon=4.4347 | ord=0.2122
    Episode  200/2000 | loss=2.7735 | hier=0.7176 | proto=0.8698 | supcon=4.2039 | ord=0.1602
    Episode  300/2000 | loss=2.6777 | hier=1.2901 | proto=1.3623 | supcon=4.1234 | ord=0.2363
    Episode  400/2000 | loss=2.6437 | hier=1.1637 | proto=1.0670 | supcon=4.3629 | ord=0.3083
    Episode  500/2000 | loss=2.3843 | hier=0.7961 | proto=0.9088 | supcon=4.1698 | ord=0.1041
    Episode  600/2000 | loss=2.3876 | hier=1.9067 | proto=1.3311 | supcon=4.0521 | ord=0.3301
    Episode  700/2000 | loss=2.4082 | hier=0.9435 | proto=1.0700 | supcon=4.2688 | ord=0.2078
    Episode  800/2000 | loss=2.3241 | hier=1.4900 | proto=1.2900 | supcon=4.2791 | o

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H9: HIERARCHICAL HYPERNETWORK RESULTS
support_protocol  k_shot  n_students  total_query_clips  mean_accuracy  sd_accuracy  mean_balanced_accuracy  sd_balanced_accuracy  mean_macro_f1  sd_macro_f1  mean_hyper_accuracy  mean_prototype_accuracy  mean_prototype_balanced_accuracy  mean_majority_accuracy  mean_majority_balanced_accuracy  mean_majority_macro_f1  mean_prototype_classes_available
           early       5          82              11016       0.683403     0.172280                0.307927              0.071349       0.199492     0.034983             0.601345                 0.617823                          0.361653                0.708270                         0.307927                0.205566                          1.902439
           early      10          82              10606       0.685759     0.173756                0.315041              0.079677       0.199843     0.035407             0.601962                 0.584436                          0.374050                0.